# 逐字 Verbatim — 雲端轉錄引擎
按 **Ctrl+F9**（Mac：⌘+F9）全部執行。跳出「這個筆記本並非由 Google 撰寫」時選 **仍要執行**。
完成後點橘色按鈕回到網站，這個分頁保持開著就好。

In [ ]:
#@title ▶ 啟動逐字 Verbatim　　（按 Ctrl+F9 或左邊的播放鍵，約 2 分鐘）
SITE = "https://bevis0414-crypto.github.io/verbatim/"
RAW = "https://raw.githubusercontent.com/bevis0414-crypto/verbatim/main/server.py"
import os, re, subprocess, sys, time, threading, urllib.request
from IPython.display import HTML, display, clear_output

def say(msg, color="#8b9098"):
    display(HTML(f'<div style="font:15px sans-serif;color:{color};margin:4px 0">{msg}</div>'))

if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
    say("⚠️ 目前沒有 GPU，轉錄會非常慢。請到上方選單「執行階段 → 變更執行階段類型」選 T4 GPU，再重新執行。", "#ff5a1f")

say("① 安裝套件中…")
subprocess.run(f"{sys.executable} -m pip install -q faster-whisper opencc-python-reimplemented python-multipart fastapi uvicorn speechbrain", shell=True)
os.makedirs("/content/verbatim", exist_ok=True)
urllib.request.urlretrieve(RAW, "/content/verbatim/server.py")
if not os.path.exists("/content/cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)

say("② 啟動轉錄引擎…")
sys.path.insert(0, "/content/verbatim")
import importlib, server
importlib.reload(server)
if not globals().get("_verbatim_server"):
    _verbatim_server = server.start(8000)
server.preload("large-v2")

say("③ 建立連線網址…")
if not globals().get("_verbatim_url"):
    proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = None
    for line in proc.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0); break
    threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()
    for _ in range(30):  # 等網址真的能連
        try:
            urllib.request.urlopen(url + "/api/system", timeout=5); break
        except Exception:
            time.sleep(2)
    _verbatim_url = url

clear_output()
link = SITE + "#server=" + _verbatim_url
display(HTML(f"""
<div style="font-family:sans-serif;background:#15171b;color:#eceae4;border-radius:16px;padding:28px;max-width:640px">
  <div style="font-size:22px;font-weight:700;margin-bottom:6px">✅ 轉錄引擎已啟動</div>
  <div style="color:#8b9098;margin-bottom:20px">點下面的按鈕回到逐字網站。<b style="color:#eceae4">這個 Colab 分頁請保持開著</b>，關掉後網站就會斷線。</div>
  <a href="{link}" target="_blank" style="display:inline-block;background:#ff5a1f;color:#fff;padding:14px 28px;border-radius:12px;font-size:17px;font-weight:600;text-decoration:none">回到逐字網站 ↗</a>
  <div style="color:#8b9098;font-size:13px;margin-top:18px">連線網址：{_verbatim_url}</div>
</div>"""))
